# 🚀 Lab 32 — Setting Up and Using a Cloud-Based Jupyter Environment

**Course:** Data Science — Week 7  
**Topic:** Cloud Computing for Data Science  
**Platform:** Google Colab (runs entirely in your browser, no installation needed!)

---

## 🎯 What You Will Learn in This Lab

1. How to launch a Jupyter Notebook in the cloud using Google Colab.
2. How to switch to a **free GPU** runtime.
3. How to mount your Google Drive to save/load files.
4. How to install Python libraries in a cloud environment.
5. How to train a real machine learning model on the cloud.
6. How to save your trained model back to Google Drive.

---

## 📋 Before You Start

✅ Open this notebook in Google Colab: https://colab.research.google.com  
✅ Sign in with a Google account.  
✅ Click **File → Save a copy in Drive** so you can edit it.  

---

**⏱️ Estimated time: 30–45 minutes**


## 🔹 Step 1 — Check Which Machine We Got From Google

When you open Colab, Google gives you a **free virtual computer** somewhere in the world.
Let's see what this machine looks like!


In [ ]:
# The "!" at the start lets us run Linux/shell commands inside a notebook cell.
# This is super useful in cloud environments.

# Command 1: Check CPU details (which processor Google gave us)
# 'cat /proc/cpuinfo' prints CPU info; 'grep "model name"' filters only the name;
# 'head -1' shows just the first line (since there are multiple cores)
!cat /proc/cpuinfo | grep "model name" | head -1

# Command 2: Check how much RAM (memory) this machine has
# '-h' means "human-readable" (shows GB instead of raw bytes)
!free -h

# Command 3: Check how much disk space we have
# '-h' again for human-readable output
!df -h / | head -2


## 🔹 Step 2 — Enable the FREE GPU 🎮

One of the best features of Colab is that Google gives you a **free GPU**.
A GPU (Graphics Processing Unit) is 10x–100x faster than a CPU for training ML models.

### 👉 How to turn it on:

1. Click the menu: **Runtime → Change runtime type**
2. Under "Hardware accelerator", select **T4 GPU** (or any GPU option).
3. Click **Save**. Colab will reconnect to a new machine with a GPU.

Then run the cell below to confirm the GPU is active.


In [ ]:
# 'nvidia-smi' is a command from NVIDIA that shows GPU information.
# If you see a table with "Tesla T4" or similar — congrats, you have a free GPU!
# If you see "command not found", go back and enable GPU in Runtime settings.

!nvidia-smi


## 🔹 Step 3 — Mount Google Drive 💾

Colab is temporary — when you close it, your files are lost.
To **save your work permanently**, we connect (mount) our Google Drive to this cloud machine.

After mounting, your Drive will appear as a folder at `/content/drive/MyDrive/`.


In [ ]:
# Import the 'drive' module from Google Colab's built-in library.
from google.colab import drive

# This line tells Colab to mount your Google Drive at the path '/content/drive'.
# When you run this, a pop-up will ask you to:
#   1. Click a link to authorize Colab.
#   2. Sign in with your Google account.
#   3. Copy a code and paste it back here.
# After that, your Drive is accessible like any regular folder!
drive.mount('/content/drive')

# Let's verify by listing files in the root of our Drive.
# The '!' runs this as a shell command (list directory contents).
!ls /content/drive/MyDrive/ | head -10


## 🔹 Step 4 — Install a New Python Library 📦

Colab has most common libraries pre-installed (numpy, pandas, scikit-learn, tensorflow, etc.).
But sometimes you need something extra. Here's how to install it.

We'll install **`joblib`** (it's already there, but this teaches you the syntax)
and **`yellowbrick`** (a nice ML visualization library you probably don't have).


In [ ]:
# 'pip' is Python's package installer.
# The '-q' flag means "quiet" — don't print every download step (keeps output clean).
# In Colab, '!pip install' is the standard way to install libraries.

!pip install -q yellowbrick

# After installing, let's import it to confirm it works:
import yellowbrick
print(f"✅ yellowbrick version {yellowbrick.__version__} installed successfully!")


## 🔹 Step 5 — Load a Real Dataset 🌸

We'll use the famous **Iris flower dataset** — it has measurements of 150 flowers
across 3 species (Setosa, Versicolor, Virginica). Classic beginner ML dataset.

This data ships with scikit-learn, so no download needed.


In [ ]:
# Import the libraries we need.
# - pandas: for data tables (like Excel in Python)
# - sklearn.datasets: has several built-in datasets for learning

import pandas as pd
from sklearn.datasets import load_iris

# Load the iris dataset into a variable.
iris = load_iris()

# Convert it into a pandas DataFrame so it looks nice and is easy to work with.
# 'iris.data' = the measurements (features)
# 'iris.feature_names' = column names like 'sepal length (cm)'
df = pd.DataFrame(iris.data, columns=iris.feature_names)

# Add a 'species' column.
# 'iris.target' is numbers (0, 1, 2); 'iris.target_names' maps them to flower names.
df['species'] = [iris.target_names[i] for i in iris.target]

# Show the first 5 rows to see what the data looks like.
df.head()


In [ ]:
# Quick exploration — how many flowers of each type?
# value_counts() counts how many times each unique value appears.
print("Flowers per species:")
print(df['species'].value_counts())

# Show basic statistics (mean, min, max, etc.) for all numeric columns.
print("\nBasic statistics:")
df.describe()


## 🔹 Step 6 — Visualize the Data 📊

Before training any model, it's a good habit to **look at the data**.
We'll use `seaborn` (pre-installed in Colab) to create a scatter plot.


In [ ]:
# Import plotting libraries.
# - matplotlib.pyplot: the core plotting library in Python
# - seaborn: prettier, statistics-focused plots built on top of matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean visual theme.
sns.set_theme(style="whitegrid")

# Create a pairplot — shows how every feature relates to every other feature,
# colored by species. Great way to see if the classes are separable.
# 'hue="species"' means color each dot by the species column.
sns.pairplot(df, hue="species", height=2)

# Add a title above the whole figure.
plt.suptitle("Iris Flower Measurements by Species", y=1.02)

# Display the plot.
plt.show()


## 🔹 Step 7 — Train a Machine Learning Model 🤖

Now the fun part! We'll train a **Random Forest Classifier** to predict the
species of a flower from its measurements. This runs entirely in the cloud.


In [ ]:
# Import the tools we need from scikit-learn.
# - train_test_split: splits data into training and testing sets
# - RandomForestClassifier: the ML model we'll use (it's like many decision trees combined)
# - accuracy_score, classification_report: to measure how good our model is

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Step 7a: Separate features (X) from the target label (y).
# X = the measurements; y = the species we want to predict.
X = df[iris.feature_names]   # all 4 measurement columns
y = df['species']             # the species column

# Step 7b: Split the data — 80% for training, 20% for testing.
# 'random_state=42' makes the split reproducible (same split every time we run).
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% of data kept for testing
    random_state=42,     # "seed" for randomness, so results are repeatable
    stratify=y           # keep the same class proportions in train and test
)

print(f"Training set size: {len(X_train)} flowers")
print(f"Testing set size:  {len(X_test)} flowers")


In [ ]:
# Step 7c: Create the model.
# n_estimators=100 means "build 100 decision trees and let them vote".
# random_state=42 again for reproducibility.
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Step 7d: Train (fit) the model on our training data.
# This is where the ML magic happens — it learns patterns from X_train to predict y_train.
model.fit(X_train, y_train)

print("✅ Model training complete!")


In [ ]:
# Step 7e: Use the trained model to predict on the test set (flowers it has never seen).
y_pred = model.predict(X_test)

# Step 7f: Measure accuracy — what fraction of predictions were correct.
accuracy = accuracy_score(y_test, y_pred)
print(f"🎯 Model Accuracy: {accuracy * 100:.2f}%")

# Step 7g: Show a detailed breakdown per species.
# 'precision' = of the ones we said were X, how many really were X
# 'recall' = of all the real X, how many did we catch
# 'f1-score' = balance of the two above
print("\nDetailed Performance:")
print(classification_report(y_test, y_pred))


## 🔹 Step 8 — Save the Trained Model to Google Drive 💾

A trained model is valuable! Let's save it to Drive so we can reuse it later
without retraining. We'll use **`joblib`** — the standard way to save scikit-learn models.


In [ ]:
#  joblib — a library that efficiently saves Python objects to disk.
import joblib
import os

# Decide where to save Importthe model.
# We'll put it in our Drive, inside a folder called 'DS_Lab_Week7'.
save_folder = '/content/drive/MyDrive/DS_Lab_Week7'

# Create the folder if it doesn't already exist.
# 'exist_ok=True' means: don't raise an error if the folder is already there.
os.makedirs(save_folder, exist_ok=True)

# Full path for the saved model file.
# '.pkl' is the standard extension for pickled (saved) Python objects.
model_path = os.path.join(save_folder, 'iris_rf_model.pkl')

# Save the model!
joblib.dump(model, model_path)

print(f"✅ Model saved successfully to: {model_path}")

# Verify the file is there by listing the folder.
!ls -lh "{save_folder}"


## 🔹 Step 9 — Load the Saved Model and Make a New Prediction 🔮

Imagine tomorrow you close Colab, go home, come back, and want to use the model
without retraining. Here's how you load it from Drive and predict on brand new data.


In [ ]:
# Load the saved model back from Drive.
loaded_model = joblib.load(model_path)
print("✅ Model loaded from Drive!")

# Make up a new flower's measurements.
# Order must match training: sepal length, sepal width, petal length, petal width (all in cm).
new_flower = [[5.1, 3.5, 1.4, 0.2]]   # double brackets because the model expects a 2D input

# Predict which species this flower is.
prediction = loaded_model.predict(new_flower)

# Also get the probability for each species (how confident the model is).
probabilities = loaded_model.predict_proba(new_flower)

print(f"\n🌸 New flower measurements: {new_flower[0]}")
print(f"🎯 Predicted species: {prediction[0]}")
print(f"\nConfidence for each species:")
for species, prob in zip(loaded_model.classes_, probabilities[0]):
    print(f"   {species:15s}: {prob * 100:.2f}%")


## 🏆 Bonus Challenge (Homework)

Try these on your own — they reinforce everything you learned today:

1. **Change the model**: Replace `RandomForestClassifier` with `LogisticRegression` (from `sklearn.linear_model`). Is accuracy higher or lower?

2. **Change the test size**: What happens to accuracy if you use `test_size=0.5` (half the data for testing)? Why?

3. **Try a different dataset**: Load the wine dataset with `from sklearn.datasets import load_wine`. Repeat Steps 5–8 for wine classification. Save the new model with a different filename.

4. **Feature importance**: Run `model.feature_importances_` — which flower measurement is the most useful for prediction? Draw a bar chart.

5. **GPU task**: Install `tensorflow` (`!pip install -q tensorflow`) and run `import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))`. Does it see the GPU?

---

## 📤 Submission

Save your completed notebook to your Drive and share the link with the instructor via the class portal.

**Filename format:** `Lab32_Cloud_YourName_RollNumber.ipynb`

---

## 🎉 Congratulations!

You just:
- ✅ Ran code on a remote Google server.
- ✅ Used a free GPU.
- ✅ Trained and saved a real ML model in the cloud.
- ✅ Loaded it back and made predictions on new data.

**This is the foundation of every real-world data science workflow.**

Keep experimenting — the cloud is your playground now. 🚀
